# Evolución de la música en Spotify (1921–2020)

**TP Grupal · Laboratorio de Métodos Cuantitativos · FCE UBA · 2C 2026**

**Integrantes:** _completar_

**Dataset:** [Spotify Dataset 1921–2020, 600k+ Tracks](https://www.kaggle.com/datasets/yamaerenay/spotify-dataset-19212020-600k-tracks) (Kaggle, usuario *yamaerenay*). Usamos el archivo `tracks.csv`, que tiene una fila por canción con su fecha de lanzamiento y sus características de audio.

### Uso de IA
Las celdas marcadas con **[Asistido por IA]** se armaron con ayuda de Claude, usando un proyecto que tenía como contexto las pautas del TP.
Prompt utilizado: «como puedo explorar el dataset»

## 3.1 Carga y exploración inicial

### a) Carga del dataset · [Asistido por IA]

Descargamos el dataset directamente desde Kaggle con la librería `kagglehub`. Así cualquiera puede volver a correr el notebook sin subir archivos a mano. No hace falta cuenta de Kaggle porque el dataset es público.

> Si la descarga falla, se puede bajar el ZIP desde la página de Kaggle, subir `tracks.csv` a Colab (ícono de carpeta a la izquierda) y cargarlo con `pd.read_csv("tracks.csv")`.

In [ ]:
%pip install -q kagglehub

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

# Descarga el dataset y devuelve la carpeta donde quedaron los archivos
path = kagglehub.dataset_download("yamaerenay/spotify-dataset-19212020-600k-tracks")
print("Archivos descargados:", os.listdir(path))

# Cargamos las canciones en un DataFrame
df = pd.read_csv(os.path.join(path, "tracks.csv"))

#### Primeras y últimas filas

Miramos el principio y el final de la tabla para ver cómo vienen los datos: qué tiene cada columna, ejemplos de valores y si hay algo raro a simple vista.

In [ ]:
display(df.head())
display(df.tail())

### b) Estructura: dimensiones, columnas y tipos de dato · [Asistido por IA]

Vemos cuántas filas (canciones) y columnas tiene el dataset, y de qué tipo es cada columna. Detalles a tener en cuenta:
- `release_date` es **texto**, no fecha. Hay que convertirla para trabajar con años.
- `artists` es un texto con forma de lista, por ejemplo `"['Uli']"`.
- `explicit` y `mode` son 0/1, y `key` es un número de 0 a 11 que representa la nota (0 = Do).

In [ ]:
print("Filas (canciones):", df.shape[0])
print("Columnas:", df.shape[1])
df.dtypes

#### Cobertura temporal

`release_date` no tiene siempre la misma precisión: algunas canciones solo tienen el año, otras año y mes, y otras la fecha completa. Primero contamos cuántas hay de cada tipo según la cantidad de caracteres. Después tomamos los primeros 4 caracteres para obtener el año y ver qué período cubre el dataset.

Por ahora el año se calcula solo para explorar. La columna definitiva se crea en la sección 3.3 (transformaciones).

El dataset **no tiene cobertura geográfica**: no incluye país de origen de las canciones ni de los artistas.

In [ ]:
# 4 caracteres = solo año, 7 = año y mes, 10 = fecha completa
precision = df["release_date"].str.len().map({4: "solo año", 7: "año y mes", 10: "fecha completa"})
print(precision.value_counts(dropna=False))

# Año de lanzamiento (temporal, solo para explorar)
anio = df["release_date"].str[:4].astype(int)
print()
print("Año mínimo:", anio.min())
print("Año máximo:", anio.max())

Contamos las canciones por década para ver si el dataset está balanceado en el tiempo.

In [ ]:
decada = (anio // 10) * 10
decada.value_counts().sort_index().rename("canciones")

Lo graficamos por año. Si las primeras décadas tienen muy pocas canciones, sus promedios van a ser menos confiables y hay que tenerlo en cuenta al sacar conclusiones.

In [ ]:
anio.value_counts().sort_index().plot(figsize=(10, 4))
plt.title("Cantidad de canciones por año de lanzamiento")
plt.xlabel("Año de lanzamiento")
plt.ylabel("Cantidad de canciones")
plt.grid(alpha=0.3)
plt.show()

### c) Calidad de los datos · [Asistido por IA]

#### Valores nulos
Contamos cuántos valores faltan en cada columna.

In [ ]:
df.isnull().sum()

#### Duplicados
Los revisamos en tres niveles:
1. **Filas idénticas**: la misma fila repetida tal cual.
2. **IDs repetidos**: el mismo identificador de Spotify más de una vez.
3. **Misma canción y artista**: mismo `name` y `artists` con distinto `id`. Suelen ser reediciones, remasters o compilados. Importan para nuestra pregunta, porque una canción de 1965 reeditada en 2010 aparece con fecha de 2010.

In [ ]:
print("Filas completamente duplicadas:", df.duplicated().sum())
print("IDs repetidos:", df["id"].duplicated().sum())
print("Misma canción y artista repetidos:", df.duplicated(subset=["name", "artists"]).sum())

In [ ]:
# Las 5 combinaciones canción + artista que más se repiten
repetidas = df[df.duplicated(subset=["name", "artists"], keep=False)]
repetidas.groupby(["name", "artists"]).size().sort_values(ascending=False).head(5)

#### Estadísticas descriptivas
Resumen de las columnas numéricas: promedio, desvío, mínimo, máximo y cuartiles. Lo transponemos (`.T`) para que cada variable quede en una fila y se lea más fácil.

In [ ]:
df.describe().T.round(2)

#### Valores sospechosos
A partir de lo que se ve en `describe()`, revisamos casos que pueden ser errores o que no son música:
- `tempo = 0` o `time_signature = 0`: Spotify no pudo medir el ritmo.
- `speechiness > 0.66`: según Spotify, es audio casi todo hablado (audiolibros, poesía, discursos).
- Duraciones extremas.

In [ ]:
print("Canciones con tempo = 0:", (df["tempo"] == 0).sum())
print("Canciones con time_signature = 0:", (df["time_signature"] == 0).sum())
print("Canciones con popularidad = 0:", (df["popularity"] == 0).sum())
print("Grabaciones casi solo habladas (speechiness > 0.66):", (df["speechiness"] > 0.66).sum())
print("Canciones de más de 20 minutos:", (df["duration_ms"] > 20 * 60 * 1000).sum())
print("Canciones de menos de 30 segundos:", (df["duration_ms"] < 30 * 1000).sum())

In [ ]:
# Ejemplos de las grabaciones más "habladas": ¿son música o no?
(df[df["speechiness"] > 0.66]
   .sort_values("speechiness", ascending=False)
   [["name", "artists", "release_date", "speechiness"]]
   .head(10))

### Hallazgos de la exploración (completar)

*Las pautas piden comentar los hallazgos antes de continuar. Completen con los números que les dio cada celda:*

- **Tamaño:** el dataset tiene ___ canciones y ___ columnas.
- **Cobertura temporal:** va de ___ a ___. Las décadas con menos canciones son ___, así que sus promedios son menos confiables.
- **Fechas:** ___ % de las canciones solo tienen el año, por eso trabajamos con el año y no con la fecha completa.
- **Nulos:** ___ (¿en qué columnas? ¿afectan a las variables que vamos a usar?).
- **Duplicados:** ___ canciones aparecen repetidas con el mismo nombre y artista (posibles reediciones).
- **Valores raros:** ___ grabaciones habladas, ___ con tempo 0, ...
- **Decisiones para la sección 3.3:** qué vamos a filtrar o transformar a partir de esto.